<a href="https://colab.research.google.com/github/EconShark/23_big_data_analysis_class/blob/main/lab02_02_pyspark_eda/Lab02_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to PySpark

3 bộ dữ liệu đi kèm:

- `adults.json`
- `salaries.csv`
- `Monthly_Transportation_Statistics.csv`

## Mục tiêu
Sinh viên có thể:
1. Khởi tạo `SparkSession`.
2. Đọc CSV/JSON và kiểm tra schema.
3. Dùng `select`, `filter`, `where`, `sort`, `groupBy`, `agg`.
4. Xử lý missing data và thao tác cột.
5. Thực hiện `join`, `union`.
6. Viết UDF cơ bản.
7. Hiểu RDD và các transformation/action.
8. Dùng Spark SQL.
9. Dùng cache/persist, broadcast và `explain()`.

## 0. Cài đặt và kiểm tra môi trường

In [4]:
# Nếu máy chưa có PySpark, bỏ dấu # ở dòng dưới:
# %pip install -q pyspark

import sys
print("Python:", sys.version)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


## 1. Khởi tạo SparkSession

`SparkSession` là điểm vào chính để làm việc với DataFrame và Spark SQL.

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IntroductionToPySpark")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master :", spark.sparkContext.master)

Spark version: 4.0.4
Spark master : local[*]


- `builder`: cấu hình phiên Spark.
- `appName(...)`: đặt tên ứng dụng.
- `master("local[*]")`: chạy local trên các CPU core.
- `getOrCreate()`: tạo mới hoặc lấy session hiện có.

## 2. Đọc JSON — `adults.json`

In [6]:
from pathlib import Path
from pyspark.sql import functions as F

DATA_DIR = Path(".")
ADULTS_FILE = str(DATA_DIR / "adults.json")

adults_df = spark.read.json(ADULTS_FILE)
adults_df.show(5, truncate=False)

+---+-------------+------+--------------+-----------------+
|age|education.num|income|marital.status|occupation       |
+---+-------------+------+--------------+-----------------+
|90 |9            |<=50K |Widowed       |?                |
|82 |9            |<=50K |Widowed       |Exec-managerial  |
|66 |10           |<=50K |Widowed       |?                |
|54 |4            |<=50K |Divorced      |Machine-op-inspct|
|41 |10           |<=50K |Separated     |Prof-specialty   |
+---+-------------+------+--------------+-----------------+
only showing top 5 rows


### 2.1 Kiểm tra schema

In [16]:
adults_df.printSchema()

root
 |-- age: long (nullable = true)
 |-- education.num: long (nullable = true)
 |-- income: string (nullable = true)
 |-- marital.status: string (nullable = true)
 |-- occupation: string (nullable = true)



Tên cột có dấu chấm như `education.num` cần dùng backtick khi tham chiếu bằng `col()`.

In [17]:
adults_df.select(
    "age",
    F.col("`education.num`").alias("education_num"),
    "occupation",
    "income"
).show(5)

+---+-------------+-----------------+------+
|age|education_num|       occupation|income|
+---+-------------+-----------------+------+
| 90|            9|                ?| <=50K|
| 82|            9|  Exec-managerial| <=50K|
| 66|           10|                ?| <=50K|
| 54|            4|Machine-op-inspct| <=50K|
| 41|           10|   Prof-specialty| <=50K|
+---+-------------+-----------------+------+
only showing top 5 rows


## 3. DataFrame cơ bản: select, filter, where, sort

In [18]:
adults_df.select("age", "occupation", "income").show(10)

+---+-----------------+------+
|age|       occupation|income|
+---+-----------------+------+
| 90|                ?| <=50K|
| 82|  Exec-managerial| <=50K|
| 66|                ?| <=50K|
| 54|Machine-op-inspct| <=50K|
| 41|   Prof-specialty| <=50K|
| 34|    Other-service| <=50K|
| 38|     Adm-clerical| <=50K|
| 74|   Prof-specialty|  >50K|
| 68|   Prof-specialty| <=50K|
| 41|     Craft-repair|  >50K|
+---+-----------------+------+
only showing top 10 rows


In [19]:
adults_over_50 = (
    adults_df
    .filter(F.col("age") > 50)
    .select("age", "occupation", "income")
)
adults_over_50.show(10)

+---+-----------------+------+
|age|       occupation|income|
+---+-----------------+------+
| 90|                ?| <=50K|
| 82|  Exec-managerial| <=50K|
| 66|                ?| <=50K|
| 54|Machine-op-inspct| <=50K|
| 74|   Prof-specialty|  >50K|
| 68|   Prof-specialty| <=50K|
| 52|    Other-service|  >50K|
| 51|                ?|  >50K|
| 57|  Exec-managerial|  >50K|
| 61|            Sales| <=50K|
+---+-----------------+------+
only showing top 10 rows


In [20]:
adults_df.where(F.col("income") == ">50K").show(10)

+---+-------------+------+--------------+----------------+
|age|education.num|income|marital.status|      occupation|
+---+-------------+------+--------------+----------------+
| 74|           16|  >50K| Never-married|  Prof-specialty|
| 41|           10|  >50K| Never-married|    Craft-repair|
| 45|           16|  >50K|      Divorced|  Prof-specialty|
| 38|           15|  >50K| Never-married|  Prof-specialty|
| 52|           13|  >50K|       Widowed|   Other-service|
| 32|           14|  >50K|     Separated| Exec-managerial|
| 51|           16|  >50K| Never-married|               ?|
| 46|           15|  >50K|      Divorced|  Prof-specialty|
| 45|            7|  >50K|      Divorced|Transport-moving|
| 57|           14|  >50K|      Divorced| Exec-managerial|
+---+-------------+------+--------------+----------------+
only showing top 10 rows


In [21]:
adults_df.orderBy(F.col("age").desc()).show(10)

+---+-------------+------+------------------+---------------+
|age|education.num|income|    marital.status|     occupation|
+---+-------------+------+------------------+---------------+
| 90|            9| <=50K|           Widowed|              ?|
| 82|            9| <=50K|           Widowed|Exec-managerial|
| 74|           16|  >50K|     Never-married| Prof-specialty|
| 73|            9| <=50K|Married-civ-spouse|Farming-fishing|
| 71|            9| <=50K|Married-civ-spouse|              ?|
| 71|            9| <=50K|Married-civ-spouse|          Sales|
| 68|            9| <=50K|          Divorced| Prof-specialty|
| 68|           10| <=50K|Married-civ-spouse|              ?|
| 67|           10| <=50K|Married-civ-spouse|              ?|
| 66|           10| <=50K|           Widowed|              ?|
+---+-------------+------+------------------+---------------+
only showing top 10 rows


### Bài tập 1
1. Lọc `age >= 40`.
2. Chỉ giữ `age`, `occupation`, `income`.
3. Sắp xếp tuổi giảm dần.
4. Hiển thị 15 dòng.

In [22]:
# TODO - Bài tập 1

## 4. Đọc CSV — `salaries.csv`

In [23]:
SALARIES_FILE = str(DATA_DIR / "salaries.csv")

salaries_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SALARIES_FILE)
)

salaries_df.show(5, truncate=False)

+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|work_year|experience_level|employment_type|job_title           |salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---------+----------------+---------------+--------------------+------+---------------+-------------+------------------+------------+----------------+------------+
|2020     |EN              |FT             |Azure Data Engineer |100000|USD            |100000       |MU                |0           |MU              |S           |
|2020     |EN              |CT             |Staff Data Analyst  |60000 |CAD            |44753        |CA                |50          |CA              |L           |
|2020     |SE              |FT             |Staff Data Scientist|164000|USD            |164000       |US                |50          |US              |M           |
|2020     

In [24]:
salaries_df.printSchema()
print("Số dòng:", salaries_df.count())
print("Số cột :", len(salaries_df.columns))

root
 |-- work_year: integer (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- salary_currency: string (nullable = true)
 |-- salary_in_usd: integer (nullable = true)
 |-- employee_residence: string (nullable = true)
 |-- remote_ratio: integer (nullable = true)
 |-- company_location: string (nullable = true)
 |-- company_size: string (nullable = true)

Số dòng: 37234
Số cột : 11


### 4.1 Select và filter

In [25]:
salaries_df.select(
    "work_year", "experience_level", "job_title",
    "salary_in_usd", "remote_ratio"
).show(10, truncate=False)

+---------+----------------+---------------------+-------------+------------+
|work_year|experience_level|job_title            |salary_in_usd|remote_ratio|
+---------+----------------+---------------------+-------------+------------+
|2020     |EN              |Azure Data Engineer  |100000       |0           |
|2020     |EN              |Staff Data Analyst   |44753        |50          |
|2020     |SE              |Staff Data Scientist |164000       |50          |
|2020     |EN              |Data Analyst         |47899        |0           |
|2020     |EX              |Data Scientist       |300000       |100         |
|2020     |MI              |Sales Data Analyst   |60000        |0           |
|2020     |EX              |Staff Data Analyst   |15000        |0           |
|2020     |MI              |Business Data Analyst|95000        |0           |
|2020     |EN              |Data Analyst         |22809        |100         |
|2020     |EN              |Data Scientist       |49268        |

In [26]:
high_salary_df = (
    salaries_df
    .filter(F.col("salary_in_usd") >= 200000)
    .select("work_year", "experience_level", "job_title", "salary_in_usd")
    .orderBy(F.col("salary_in_usd").desc())
)
high_salary_df.show(20, truncate=False)

+---------+----------------+--------------------------+-------------+
|work_year|experience_level|job_title                 |salary_in_usd|
+---------+----------------+--------------------------+-------------+
|2024     |MI              |AI Architect              |800000       |
|2024     |EN              |Data Analyst              |774000       |
|2023     |MI              |Machine Learning Scientist|750000       |
|2023     |MI              |Machine Learning Engineer |750000       |
|2023     |MI              |Data Engineer             |750000       |
|2023     |SE              |Data Scientist            |750000       |
|2024     |SE              |Data Scientist            |750000       |
|2024     |SE              |Analytics Engineer        |750000       |
|2024     |MI              |Machine Learning Scientist|750000       |
|2024     |SE              |Data Analyst              |750000       |
|2024     |MI              |Machine Learning Scientist|750000       |
|2024     |SE       

## 5. Aggregation: groupBy và agg

In [27]:
from pyspark.sql.functions import avg, min, max, count, round as spark_round

salary_by_experience = (
    salaries_df
    .groupBy("experience_level")
    .agg(
        count("*").alias("n"),
        spark_round(avg("salary_in_usd"), 2).alias("avg_salary_usd"),
        min("salary_in_usd").alias("min_salary_usd"),
        max("salary_in_usd").alias("max_salary_usd")
    )
    .orderBy("experience_level")
)
salary_by_experience.show()

+----------------+-----+--------------+--------------+--------------+
|experience_level|    n|avg_salary_usd|min_salary_usd|max_salary_usd|
+----------------+-----+--------------+--------------+--------------+
|              EN| 3166|     107310.08|         15000|        774000|
|              EX|  822|     198208.34|         15000|        526400|
|              MI|10723|     144187.63|         15000|        800000|
|              SE|22523|     174433.86|         15809|        750000|
+----------------+-----+--------------+--------------+--------------+



In [28]:
(
    salaries_df
    .groupBy("work_year")
    .agg(
        count("*").alias("n"),
        spark_round(avg("salary_in_usd"), 2).alias("avg_salary_usd")
    )
    .orderBy("work_year")
    .show()
)

+---------+-----+--------------+
|work_year|    n|avg_salary_usd|
+---------+-----+--------------+
|     2020|   75|     102250.87|
|     2021|  218|      99922.07|
|     2022| 1658|     134215.09|
|     2023| 8522|     153700.76|
|     2024|26761|     165006.93|
+---------+-----+--------------+



### Bài tập 2
Tính theo `company_size`: số bản ghi, lương trung bình, lương lớn nhất.
Sắp xếp theo lương trung bình giảm dần.

In [29]:
# TODO - Bài tập 2

## 6. Missing data

In [30]:
null_counts = salaries_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in salaries_df.columns
])
null_counts.show(truncate=False)

+---------+----------------+---------------+---------+------+---------------+-------------+------------------+------------+----------------+------------+
|work_year|experience_level|employment_type|job_title|salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+---------+----------------+---------------+---------+------+---------------+-------------+------------------+------------+----------------+------------+
|0        |0               |0              |0        |0     |0              |0            |0                 |0           |0               |0           |
+---------+----------------+---------------+---------+------+---------------+-------------+------------------+------------+----------------+------------+



In [31]:
salaries_dropna = salaries_df.na.drop()
print("Trước:", salaries_df.count())
print("Sau :", salaries_dropna.count())

Trước: 37234
Sau : 37234


Trong `adults.json`, một số giá trị thiếu của `occupation` được ký hiệu bằng `"?"`.
Chuẩn hóa `"?"` thành `null` trước khi xử lý.

In [32]:
adults_clean = adults_df.withColumn(
    "occupation",
    F.when(F.col("occupation") == "?", None).otherwise(F.col("occupation"))
)

adults_clean.filter(F.col("occupation").isNull()).show(10)

+---+-------------+------+------------------+----------+
|age|education.num|income|    marital.status|occupation|
+---+-------------+------+------------------+----------+
| 90|            9| <=50K|           Widowed|      NULL|
| 66|           10| <=50K|           Widowed|      NULL|
| 51|           16|  >50K|     Never-married|      NULL|
| 61|            9| <=50K|Married-civ-spouse|      NULL|
| 71|            9| <=50K|Married-civ-spouse|      NULL|
| 68|           10| <=50K|Married-civ-spouse|      NULL|
| 67|           10| <=50K|Married-civ-spouse|      NULL|
| 41|           11|  >50K|Married-civ-spouse|      NULL|
+---+-------------+------+------------------+----------+



In [33]:
adults_filled = adults_clean.na.fill({"occupation": "Unknown"})
adults_filled.show(10)

+---+-------------+------+--------------+-----------------+
|age|education.num|income|marital.status|       occupation|
+---+-------------+------+--------------+-----------------+
| 90|            9| <=50K|       Widowed|          Unknown|
| 82|            9| <=50K|       Widowed|  Exec-managerial|
| 66|           10| <=50K|       Widowed|          Unknown|
| 54|            4| <=50K|      Divorced|Machine-op-inspct|
| 41|           10| <=50K|     Separated|   Prof-specialty|
| 34|            9| <=50K|      Divorced|    Other-service|
| 38|            6| <=50K|     Separated|     Adm-clerical|
| 74|           16|  >50K| Never-married|   Prof-specialty|
| 68|            9| <=50K|      Divorced|   Prof-specialty|
| 41|           10|  >50K| Never-married|     Craft-repair|
+---+-------------+------+--------------+-----------------+
only showing top 10 rows


## 7. Thao tác cột: withColumn, rename, drop

In [34]:
salary_features = (
    salaries_df
    .withColumn("salary_k_usd", F.round(F.col("salary_in_usd") / 1000, 2))
    .withColumn(
        "remote_type",
        F.when(F.col("remote_ratio") == 100, "Remote")
         .when(F.col("remote_ratio") == 50, "Hybrid")
         .otherwise("On-site")
    )
)

salary_features.select(
    "job_title", "salary_in_usd", "salary_k_usd",
    "remote_ratio", "remote_type"
).show(10, truncate=False)

+---------------------+-------------+------------+------------+-----------+
|job_title            |salary_in_usd|salary_k_usd|remote_ratio|remote_type|
+---------------------+-------------+------------+------------+-----------+
|Azure Data Engineer  |100000       |100.0       |0           |On-site    |
|Staff Data Analyst   |44753        |44.75       |50          |Hybrid     |
|Staff Data Scientist |164000       |164.0       |50          |Hybrid     |
|Data Analyst         |47899        |47.9        |0           |On-site    |
|Data Scientist       |300000       |300.0       |100         |Remote     |
|Sales Data Analyst   |60000        |60.0        |0           |On-site    |
|Staff Data Analyst   |15000        |15.0        |0           |On-site    |
|Business Data Analyst|95000        |95.0        |0           |On-site    |
|Data Analyst         |22809        |22.81       |100         |Remote     |
|Data Scientist       |49268        |49.27       |0           |On-site    |
+-----------

In [35]:
renamed_df = salary_features.withColumnRenamed("salary_in_usd", "salary_usd")
renamed_df.select("job_title", "salary_usd").show(5, truncate=False)

+--------------------+----------+
|job_title           |salary_usd|
+--------------------+----------+
|Azure Data Engineer |100000    |
|Staff Data Analyst  |44753     |
|Staff Data Scientist|164000    |
|Data Analyst        |47899     |
|Data Scientist      |300000    |
+--------------------+----------+
only showing top 5 rows


In [36]:
dropped_df = salary_features.drop("salary", "salary_currency")
print(dropped_df.columns)

['work_year', 'experience_level', 'employment_type', 'job_title', 'salary_in_usd', 'employee_residence', 'remote_ratio', 'company_location', 'company_size', 'salary_k_usd', 'remote_type']


## 8. Join

In [37]:
experience_lookup = spark.createDataFrame(
    [
        ("EN", "Entry-level / Junior"),
        ("MI", "Mid-level / Intermediate"),
        ("SE", "Senior / Expert"),
        ("EX", "Executive-level / Director"),
    ],
    ["experience_level", "experience_label"]
)

experience_lookup.show()

+----------------+--------------------+
|experience_level|    experience_label|
+----------------+--------------------+
|              EN|Entry-level / Junior|
|              MI|Mid-level / Inter...|
|              SE|     Senior / Expert|
|              EX|Executive-level /...|
+----------------+--------------------+



In [38]:
salary_joined = salaries_df.join(
    experience_lookup,
    on="experience_level",
    how="left"
)

salary_joined.select(
    "experience_level", "experience_label",
    "job_title", "salary_in_usd"
).show(10, truncate=False)

+----------------+--------------------------+---------------------+-------------+
|experience_level|experience_label          |job_title            |salary_in_usd|
+----------------+--------------------------+---------------------+-------------+
|EX              |Executive-level / Director|Data Scientist       |300000       |
|EX              |Executive-level / Director|Staff Data Analyst   |15000        |
|MI              |Mid-level / Intermediate  |Sales Data Analyst   |60000        |
|MI              |Mid-level / Intermediate  |Business Data Analyst|95000        |
|EN              |Entry-level / Junior      |Azure Data Engineer  |100000       |
|EN              |Entry-level / Junior      |Staff Data Analyst   |44753        |
|EN              |Entry-level / Junior      |Data Analyst         |47899        |
|EN              |Entry-level / Junior      |Data Analyst         |22809        |
|EN              |Entry-level / Junior      |Data Scientist       |49268        |
|SE             

## 9. Union

In [39]:
sample_a = salaries_df.filter(F.col("work_year") == 2020).limit(5)
sample_b = salaries_df.filter(F.col("work_year") == 2021).limit(5)

sample_union = sample_a.union(sample_b)
print("Số dòng sau union:", sample_union.count())

sample_union.select("work_year", "job_title", "salary_in_usd").show(truncate=False)

Số dòng sau union: 10
+---------+-------------------------------+-------------+
|work_year|job_title                      |salary_in_usd|
+---------+-------------------------------+-------------+
|2020     |Azure Data Engineer            |100000       |
|2020     |Staff Data Analyst             |44753        |
|2020     |Staff Data Scientist           |164000       |
|2020     |Data Analyst                   |47899        |
|2020     |Data Scientist                 |300000       |
|2021     |Staff Machine Learning Engineer|185000       |
|2021     |Business Data Analyst          |56000        |
|2021     |Machine Learning Developer     |60000        |
|2021     |Data Scientist                 |150000       |
|2021     |AI Scientist                   |30000        |
+---------+-------------------------------+-------------+



## 10. UDF — User Defined Function

Trong thực tế, nên ưu tiên các hàm có sẵn trong `pyspark.sql.functions`
trước khi dùng Python UDF.

In [40]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

def salary_band(salary):
    if salary is None:
        return None
    if salary < 50000:
        return "Low"
    elif salary < 100000:
        return "Medium"
    elif salary < 200000:
        return "High"
    return "Very High"

salary_band_udf = udf(salary_band, StringType())

salary_udf_df = salaries_df.withColumn(
    "salary_band",
    salary_band_udf(F.col("salary_in_usd"))
)

salary_udf_df.select(
    "job_title", "salary_in_usd", "salary_band"
).show(10, truncate=False)

+---------------------+-------------+-----------+
|job_title            |salary_in_usd|salary_band|
+---------------------+-------------+-----------+
|Azure Data Engineer  |100000       |High       |
|Staff Data Analyst   |44753        |Low        |
|Staff Data Scientist |164000       |High       |
|Data Analyst         |47899        |Low        |
|Data Scientist       |300000       |Very High  |
|Sales Data Analyst   |60000        |Medium     |
|Staff Data Analyst   |15000        |Low        |
|Business Data Analyst|95000        |Medium     |
|Data Analyst         |22809        |Low        |
|Data Scientist       |49268        |Low        |
+---------------------+-------------+-----------+
only showing top 10 rows


### 10.1 Cách ưu tiên hơn: Spark native expression

In [41]:
salary_native_df = salaries_df.withColumn(
    "salary_band",
    F.when(F.col("salary_in_usd") < 50000, "Low")
     .when(F.col("salary_in_usd") < 100000, "Medium")
     .when(F.col("salary_in_usd") < 200000, "High")
     .otherwise("Very High")
)

salary_native_df.select(
    "job_title", "salary_in_usd", "salary_band"
).show(10, truncate=False)

+---------------------+-------------+-----------+
|job_title            |salary_in_usd|salary_band|
+---------------------+-------------+-----------+
|Azure Data Engineer  |100000       |High       |
|Staff Data Analyst   |44753        |Low        |
|Staff Data Scientist |164000       |High       |
|Data Analyst         |47899        |Low        |
|Data Scientist       |300000       |Very High  |
|Sales Data Analyst   |60000        |Medium     |
|Staff Data Analyst   |15000        |Low        |
|Business Data Analyst|95000        |Medium     |
|Data Analyst         |22809        |Low        |
|Data Scientist       |49268        |Low        |
+---------------------+-------------+-----------+
only showing top 10 rows


## 11. RDD — Resilient Distributed Dataset

- Transformation: `map`, `filter`, `flatMap`, `reduceByKey`, ...
- Action: `collect`, `count`, `take`, `reduce`, ...

> Không nên `collect()` toàn bộ dữ liệu lớn về driver.

In [42]:
adults_rdd = adults_df.rdd

print("Kiểu:", type(adults_rdd))
print("Số phần tử:", adults_rdd.count())

for row in adults_rdd.take(3):
    print(row)

Kiểu: <class 'pyspark.core.rdd.RDD'>
Số phần tử: 99
Row(age=90, education.num=9, income='<=50K', marital.status='Widowed', occupation='?')
Row(age=82, education.num=9, income='<=50K', marital.status='Widowed', occupation='Exec-managerial')
Row(age=66, education.num=10, income='<=50K', marital.status='Widowed', occupation='?')


In [43]:
ages_rdd = adults_rdd.map(lambda row: row["age"])
print("5 tuổi đầu:", ages_rdd.take(5))

5 tuổi đầu: [90, 82, 66, 54, 41]


In [44]:
older_rdd = adults_rdd.filter(lambda row: row["age"] >= 60)
print("Số người >= 60:", older_rdd.count())
print(older_rdd.take(5))

Số người >= 60: 17
[Row(age=90, education.num=9, income='<=50K', marital.status='Widowed', occupation='?'), Row(age=82, education.num=9, income='<=50K', marital.status='Widowed', occupation='Exec-managerial'), Row(age=66, education.num=10, income='<=50K', marital.status='Widowed', occupation='?'), Row(age=74, education.num=16, income='>50K', marital.status='Never-married', occupation='Prof-specialty'), Row(age=68, education.num=9, income='<=50K', marital.status='Divorced', occupation='Prof-specialty')]


In [45]:
income_count_rdd = (
    adults_rdd
    .map(lambda row: (row["income"], 1))
    .reduceByKey(lambda x, y: x + y)
)

print(income_count_rdd.collect())

[('<=50K', 20), ('>50K', 79)]


## 12. DataFrame vs RDD

| DataFrame | RDD |
|---|---|
| API mức cao | API mức thấp |
| Có schema | Không có schema dạng bảng rõ ràng |
| Hỗ trợ Spark SQL | Không trực tiếp |
| Có optimizer | Ít tối ưu tự động hơn |
| Phù hợp phần lớn ETL/analytics | Dùng khi cần xử lý mức thấp |

**Khuyến nghị:** ưu tiên DataFrame cho phần lớn bài toán phân tích dữ liệu.

## 13. Spark SQL

In [46]:
salaries_df.createOrReplaceTempView("salaries")

spark.sql("""
SELECT
    experience_level,
    COUNT(*) AS n,
    ROUND(AVG(salary_in_usd), 2) AS avg_salary_usd
FROM salaries
GROUP BY experience_level
ORDER BY avg_salary_usd DESC
""").show()

+----------------+-----+--------------+
|experience_level|    n|avg_salary_usd|
+----------------+-----+--------------+
|              EX|  822|     198208.34|
|              SE|22523|     174433.86|
|              MI|10723|     144187.63|
|              EN| 3166|     107310.08|
+----------------+-----+--------------+



### 13.1 Kết hợp SQL và DataFrame API

In [47]:
sql_result = spark.sql("""
SELECT *
FROM salaries
WHERE salary_in_usd >= 150000
""")

(
    sql_result
    .withColumn("salary_k_usd", F.round(F.col("salary_in_usd") / 1000, 1))
    .select("job_title", "experience_level", "salary_in_usd", "salary_k_usd")
    .orderBy(F.col("salary_in_usd").desc())
    .show(15, truncate=False)
)

+--------------------------+----------------+-------------+------------+
|job_title                 |experience_level|salary_in_usd|salary_k_usd|
+--------------------------+----------------+-------------+------------+
|AI Architect              |MI              |800000       |800.0       |
|Data Analyst              |EN              |774000       |774.0       |
|Machine Learning Scientist|MI              |750000       |750.0       |
|Machine Learning Engineer |MI              |750000       |750.0       |
|Data Engineer             |MI              |750000       |750.0       |
|Data Scientist            |SE              |750000       |750.0       |
|Data Scientist            |SE              |750000       |750.0       |
|Analytics Engineer        |SE              |750000       |750.0       |
|Machine Learning Scientist|MI              |750000       |750.0       |
|Data Analyst              |SE              |750000       |750.0       |
|Machine Learning Scientist|MI              |750000

### Bài tập 3 — Spark SQL
Tìm 10 `job_title` có:
- ít nhất 20 bản ghi,
- lương trung bình cao nhất.

Gợi ý: `GROUP BY job_title`, `HAVING COUNT(*) >= 20`.

In [48]:
# TODO - Bài tập 3

## 14. Dữ liệu giao thông nhiều cột

Dataset này phù hợp để luyện chọn cột, missing data, parsing thời gian,
cache/persist và execution plan.

In [7]:
TRANSPORT_FILE = str(DATA_DIR / "Monthly_Transportation_Statistics.csv")

transport_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(TRANSPORT_FILE)
)

print("Số dòng:", transport_df.count())
print("Số cột :", len(transport_df.columns))
transport_df.select("Index", "Date").show(5, truncate=False)

Số dòng: 924
Số cột : 136
+-----+----------------------+
|Index|Date                  |
+-----+----------------------+
|0    |01/01/1947 12:00:00 AM|
|1    |02/01/1947 12:00:00 AM|
|2    |03/01/1947 12:00:00 AM|
|3    |04/01/1947 12:00:00 AM|
|4    |05/01/1947 12:00:00 AM|
+-----+----------------------+
only showing top 5 rows


In [8]:
transport_small = transport_df.select(
    "Index",
    "Date",
    "Highway Fatalities",
    "Highway Fuel Price - Regular Gasoline",
    "Unemployment Rate - Seasonally Adjusted",
    "`U.S. Airline Traffic - Total - Seasonally Adjusted`"
)

transport_small.show(10, truncate=False)

+-----+----------------------+------------------+-------------------------------------+---------------------------------------+--------------------------------------------------+
|Index|Date                  |Highway Fatalities|Highway Fuel Price - Regular Gasoline|Unemployment Rate - Seasonally Adjusted|U.S. Airline Traffic - Total - Seasonally Adjusted|
+-----+----------------------+------------------+-------------------------------------+---------------------------------------+--------------------------------------------------+
|0    |01/01/1947 12:00:00 AM|NULL              |NULL                                 |NULL                                   |NULL                                              |
|1    |02/01/1947 12:00:00 AM|NULL              |NULL                                 |NULL                                   |NULL                                              |
|2    |03/01/1947 12:00:00 AM|NULL              |NULL                                 |NULL              

In [11]:
transport_nulls = transport_small.select([
    F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c)
    for c in transport_small.columns
])

transport_nulls.show(truncate=False)

+-----+----+------------------+-------------------------------------+---------------------------------------+--------------------------------------------------+
|Index|Date|Highway Fatalities|Highway Fuel Price - Regular Gasoline|Unemployment Rate - Seasonally Adjusted|U.S. Airline Traffic - Total - Seasonally Adjusted|
+-----+----+------------------+-------------------------------------+---------------------------------------+--------------------------------------------------+
|0    |0   |869               |536                                  |707                                    |857                                               |
+-----+----+------------------+-------------------------------------+---------------------------------------+--------------------------------------------------+



In [12]:
transport_typed = transport_small.withColumn(
    "DateParsed",
    F.to_timestamp("Date", "MM/dd/yyyy hh:mm:ss a")
)

transport_typed.select("Date", "DateParsed").show(5, truncate=False)

+----------------------+-------------------+
|Date                  |DateParsed         |
+----------------------+-------------------+
|01/01/1947 12:00:00 AM|1947-01-01 00:00:00|
|02/01/1947 12:00:00 AM|1947-02-01 00:00:00|
|03/01/1947 12:00:00 AM|1947-03-01 00:00:00|
|04/01/1947 12:00:00 AM|1947-04-01 00:00:00|
|05/01/1947 12:00:00 AM|1947-05-01 00:00:00|
+----------------------+-------------------+
only showing top 5 rows


## 15. Execution plan và lazy evaluation

Spark dùng lazy evaluation: transformation chưa nhất thiết chạy ngay.
Action như `show()`, `count()`, `collect()`, `write...` mới kích hoạt job.

In [53]:
plan_df = (
    salaries_df
    .filter(F.col("salary_in_usd") > 100000)
    .select("experience_level", "job_title", "salary_in_usd")
)

plan_df.explain(mode="formatted")

== Physical Plan ==
* Filter (2)
+- Scan csv  (1)


(1) Scan csv 
Output [3]: [experience_level#345, job_title#347, salary_in_usd#350]
Batched: false
Location: InMemoryFileIndex [file:/content/salaries.csv]
PushedFilters: [IsNotNull(salary_in_usd), GreaterThan(salary_in_usd,100000)]
ReadSchema: struct<experience_level:string,job_title:string,salary_in_usd:int>

(2) Filter [codegen id : 1]
Input [3]: [experience_level#345, job_title#347, salary_in_usd#350]
Condition : (isnotnull(salary_in_usd#350) AND (salary_in_usd#350 > 100000))




## 16. Cache và Persist

In [54]:
cached_salary = salaries_df.cache()

print("Rows:", cached_salary.count())
cached_salary.groupBy("experience_level").avg("salary_in_usd").show()
cached_salary.groupBy("company_size").count().show()

cached_salary.unpersist()

Rows: 37234
+----------------+------------------+
|experience_level|avg(salary_in_usd)|
+----------------+------------------+
|              EX|198208.34306569342|
|              MI|144187.63228574092|
|              EN|107310.08243840809|
|              SE|174433.86120854237|
+----------------+------------------+

+------------+-----+
|company_size|count|
+------------+-----+
|           L| 1451|
|           M|35583|
|           S|  200|
+------------+-----+



DataFrame[work_year: int, experience_level: string, employment_type: string, job_title: string, salary: int, salary_currency: string, salary_in_usd: int, employee_residence: string, remote_ratio: int, company_location: string, company_size: string]

In [55]:
from pyspark import StorageLevel

transport_cached = transport_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Rows:", transport_cached.count())
transport_cached.select("Date", "Highway Fatalities").show(5)

transport_cached.unpersist()

Rows: 924
+--------------------+------------------+
|                Date|Highway Fatalities|
+--------------------+------------------+
|01/01/1947 12:00:...|              NULL|
|02/01/1947 12:00:...|              NULL|
|03/01/1947 12:00:...|              NULL|
|04/01/1947 12:00:...|              NULL|
|05/01/1947 12:00:...|              NULL|
+--------------------+------------------+
only showing top 5 rows


DataFrame[Index: int, Date: string, Air Safety - General Aviation Fatalities: int, Highway Fatalities Per 100 Million Vehicle Miles Traveled: double, Highway Fatalities: int, U.S. Airline Traffic - Total - Seasonally Adjusted: double, U.S. Airline Traffic - International - Seasonally Adjusted: double, U.S. Airline Traffic - Domestic - Seasonally Adjusted: double, Transit Ridership - Other Transit Modes - Adjusted: int, Transit Ridership - Fixed Route Bus - Adjusted: int, Transit Ridership - Urban Rail - Adjusted: int, Freight Rail Intermodal Units: int, Freight Rail Carloads: int, Highway Vehicle Miles Traveled - All Systems: double, Highway Vehicle Miles Traveled - Total Rural: bigint, Highway Vehicle Miles Traveled - Other Rural: bigint, Highway Vehicle Miles Traveled - Rural Other Arterial: bigint, Highway Vehicle Miles Traveled - Rural Interstate: bigint, State and Local Government Construction Spending - Breakwater/Jetty: int, State and Local Government Construction Spending - Dam

## 17. Broadcast Join

In [56]:
from pyspark.sql.functions import broadcast

broadcast_joined = salaries_df.join(
    broadcast(experience_lookup),
    on="experience_level",
    how="left"
)

broadcast_joined.select(
    "experience_level", "experience_label", "job_title"
).show(10, truncate=False)

+----------------+--------------------------+---------------------+
|experience_level|experience_label          |job_title            |
+----------------+--------------------------+---------------------+
|EN              |Entry-level / Junior      |Azure Data Engineer  |
|EN              |Entry-level / Junior      |Staff Data Analyst   |
|SE              |Senior / Expert           |Staff Data Scientist |
|EN              |Entry-level / Junior      |Data Analyst         |
|EX              |Executive-level / Director|Data Scientist       |
|MI              |Mid-level / Intermediate  |Sales Data Analyst   |
|EX              |Executive-level / Director|Staff Data Analyst   |
|MI              |Mid-level / Intermediate  |Business Data Analyst|
|EN              |Entry-level / Junior      |Data Analyst         |
|EN              |Entry-level / Junior      |Data Scientist       |
+----------------+--------------------------+---------------------+
only showing top 10 rows


## 18. Mini Lab — Phân tích dữ liệu lương bằng PySpark

1. Dataset có bao nhiêu dòng?
2. Có bao nhiêu `job_title` khác nhau?
3. Top 10 `job_title` có nhiều bản ghi nhất.
4. Lương trung bình theo `experience_level`.
5. Lương trung bình theo `remote_ratio`.
6. Top 10 `job_title` có lương trung bình cao nhất, chỉ xét chức danh có ít nhất 30 bản ghi.
7. Làm câu 6 bằng DataFrame API và Spark SQL.

In [57]:
print("Số dòng:", salaries_df.count())
print(
    "Số job title khác nhau:",
    salaries_df.select("job_title").distinct().count()
)

Số dòng: 37234
Số job title khác nhau: 215


In [58]:
(
    salaries_df
    .groupBy("job_title")
    .count()
    .orderBy(F.col("count").desc())
    .show(10, truncate=False)
)

+-------------------------+-----+
|job_title                |count|
+-------------------------+-----+
|Data Scientist           |7448 |
|Data Engineer            |6103 |
|Data Analyst             |4351 |
|Machine Learning Engineer|3990 |
|Software Engineer        |2935 |
|Research Scientist       |1569 |
|Applied Scientist        |881  |
|Data Architect           |807  |
|Analytics Engineer       |758  |
|Research Engineer        |708  |
+-------------------------+-----+
only showing top 10 rows


In [59]:
(
    salaries_df
    .groupBy("experience_level")
    .agg(F.round(F.avg("salary_in_usd"), 2).alias("avg_salary_usd"))
    .orderBy(F.col("avg_salary_usd").desc())
    .show()
)

+----------------+--------------+
|experience_level|avg_salary_usd|
+----------------+--------------+
|              EX|     198208.34|
|              SE|     174433.86|
|              MI|     144187.63|
|              EN|     107310.08|
+----------------+--------------+



In [60]:
(
    salaries_df
    .groupBy("remote_ratio")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("salary_in_usd"), 2).alias("avg_salary_usd")
    )
    .orderBy("remote_ratio")
    .show()
)

+------------+-----+--------------+
|remote_ratio|    n|avg_salary_usd|
+------------+-----+--------------+
|           0|28468|     164451.77|
|          50|  257|      82145.09|
|         100| 8509|     149823.09|
+------------+-----+--------------+



In [61]:
(
    salaries_df
    .groupBy("job_title")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("salary_in_usd"), 2).alias("avg_salary_usd")
    )
    .filter(F.col("n") >= 30)
    .orderBy(F.col("avg_salary_usd").desc())
    .show(10, truncate=False)
)

+----------------------------+----+--------------+
|job_title                   |n   |avg_salary_usd|
+----------------------------+----+--------------+
|Engineering Manager         |136 |281769.24     |
|AI Architect                |65  |230604.4      |
|Head of Data                |101 |226601.37     |
|Prompt Engineer             |31  |208761.0      |
|Research Engineer           |708 |203448.49     |
|Backend Engineer            |50  |202459.58     |
|Site Reliability Engineer   |100 |199818.93     |
|Product Manager             |220 |198923.43     |
|Research Scientist          |1569|198505.56     |
|Data Infrastructure Engineer|42  |198314.88     |
+----------------------------+----+--------------+
only showing top 10 rows


In [62]:
spark.sql("""
SELECT
    job_title,
    COUNT(*) AS n,
    ROUND(AVG(salary_in_usd), 2) AS avg_salary_usd
FROM salaries
GROUP BY job_title
HAVING COUNT(*) >= 30
ORDER BY avg_salary_usd DESC
LIMIT 10
""").show(truncate=False)

+----------------------------+----+--------------+
|job_title                   |n   |avg_salary_usd|
+----------------------------+----+--------------+
|Engineering Manager         |136 |281769.24     |
|AI Architect                |65  |230604.4      |
|Head of Data                |101 |226601.37     |
|Prompt Engineer             |31  |208761.0      |
|Research Engineer           |708 |203448.49     |
|Backend Engineer            |50  |202459.58     |
|Site Reliability Engineer   |100 |199818.93     |
|Product Manager             |220 |198923.43     |
|Research Scientist          |1569|198505.56     |
|Data Infrastructure Engineer|42  |198314.88     |
+----------------------------+----+--------------+



## 19. Bài tập tự luyện

### A. Adults
1. Đếm theo `income`.
2. Tuổi trung bình theo `income`.
3. Đếm theo `marital.status`.
4. Chuẩn hóa `occupation = "?"` thành null.
5. Tìm 5 occupation phổ biến nhất.

### B. Salaries
1. Top 10 chức danh có lương trung bình cao nhất.
2. Lương trung bình theo `company_size`.
3. Lương trung bình theo `work_year` và `experience_level`.
4. Tạo cột phân nhóm mức lương.
5. So sánh Remote/Hybrid/On-site.

### C. Transportation
1. Chọn 8–10 cột có ý nghĩa.
2. Đếm missing value.
3. Chỉ giữ dữ liệu từ năm 2000.
4. Tính trung bình theo năm cho một chỉ số.
5. Giải thích execution plan.

## 20. Cheatsheet

```python
# Đọc dữ liệu
spark.read.csv(...)
spark.read.json(...)
spark.read.parquet(...)

# Khám phá
df.show()
df.printSchema()
df.count()

# DataFrame
df.select(...)
df.filter(...)
df.where(...)
df.orderBy(...)

# Missing
df.na.drop()
df.na.fill({...})
df.filter(F.col("x").isNotNull())

# Cột
df.withColumn(...)
df.withColumnRenamed(...)
df.drop(...)

# Tổng hợp
df.groupBy(...).agg(...)

# Kết hợp
df1.join(df2, ..., how="inner")
df1.union(df2)

# SQL
df.createOrReplaceTempView("table")
spark.sql("SELECT ...")

# RDD
df.rdd
rdd.map(...)
rdd.filter(...)
rdd.reduceByKey(...)
rdd.take(...)

# Hiệu năng
df.explain()
df.cache()
df.persist(...)
df.unpersist()
broadcast(df)
```

## 21. Kết thúc SparkSession

In [63]:
# spark.stop()